In [4]:
# Install ADK and LiteLLM
!pip install google-adk -q
!pip install google-adk[extensions] -q
!pip install litellm -q
!pip install nest_asyncio

print("dependencies installed...")

dependencies installed...


In [11]:
import os
from getpass import getpass

# Get inputs
PROJECT_ID = getpass("Enter your GCP project id: ")
GEMINI_API_KEY = getpass("Enter your Google Gemini API key: ")

# Set environment variables so LiteLLM and your functions automatically find them
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("Credentials loaded successfully into environment!")

Enter your GCP project id: ··········
Enter your Google Gemini API key: ··········
Credentials loaded successfully into environment!


In [12]:
import os
import asyncio
import logging
import warnings
import nest_asyncio
from datetime import datetime
from typing import Dict, Any, Optional

# ---------------------------------------------------------------------------
# Suppress ADK agent deprecation warnings
# ---------------------------------------------------------------------------
warnings.filterwarnings("ignore", category=DeprecationWarning, module="google.adk")

from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import ToolContext, AgentTool, google_search
from google.adk.models import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types
from google.genai.types import Content, Part

# Apply nest_asyncio for Jupyter / Colab event loops
nest_asyncio.apply()

# Configure logging
logging.basicConfig(level=logging.INFO)

MODEL_NAME = "gemini-3.6-flash"
RETRY_OPTIONS = types.HttpRetryOptions(initial_delay=1, max_delay=3, attempts=5)

# ============================================================================
# 1. Callback Functions & Custom State Tools
# ============================================================================

def logging_before_callback(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Logs agent execution activity before invocation."""
    print(f"\n⚙️ [Callback] Invoking agent: {callback_context.agent_name}")
    return None


def append_to_state(
    tool_context: ToolContext, field: str, response: str
) -> Dict[str, str]:
    """Appends text output to an existing state list key in tool context."""
    existing_state = tool_context.state.get(field, [])
    if isinstance(existing_state, str):
        existing_state = [existing_state]
    tool_context.state[field] = existing_state + [response]
    logging.info(f"[State Updated: '{field}'] {response[:100]}...")
    return {"status": "success"}


# Dedicated search sub-agent isolating google_search
search_sub_agent = Agent(
    name="search_sub_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Performs live web searches using Google Search.",
    instruction="Use `google_search` to find up-to-date facts relative to the current date specified in the user prompt context.",
    tools=[google_search],
    before_model_callback=logging_before_callback,
)

# ============================================================================
# 2. Pipeline Agents: Search -> Critique -> Refine
# ============================================================================

# Step 1: Search Agent (Drafts initial answer using live web search)
search_agent = Agent(
    name="search_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Searches for up-to-date data to answer the user query.",
    instruction="""
    USER_QUERY: { PROMPT? }

    INSTRUCTIONS:
    - Analyze USER_QUERY and pay close attention to the SYSTEM CONTEXT date provided in the prompt.
    - Call `search_sub_agent` to query current web information relative to that date.
    - Draft an initial detailed answer.
    - Call `append_to_state` to store your draft in 'INITIAL_RESPONSE'.
    """,
    tools=[AgentTool(agent=search_sub_agent), append_to_state],
    before_model_callback=logging_before_callback,
)

# Step 2: Critique Agent (Evaluates initial response and suggests improvements)
critique_agent = Agent(
    name="critique_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Critiques the initial response for accuracy, date alignment, and completeness.",
    instruction="""
    USER_QUERY: { PROMPT? }
    INITIAL_RESPONSE: { INITIAL_RESPONSE? }

    INSTRUCTIONS:
    - Review INITIAL_RESPONSE against USER_QUERY.
    - Check the current date mentioned in USER_QUERY's SYSTEM CONTEXT and ensure all events/dates match that timeframe (and are not from past years).
    - Provide specific critique notes and constructive suggestions for improvement.
    - Call `append_to_state` to store your feedback in 'CRITIQUE_FEEDBACK'.
    """,
    tools=[append_to_state],
    before_model_callback=logging_before_callback,
)

# Step 3: Refine Agent (Rewrites and outputs the final response based on critique)
refine_agent = Agent(
    name="refine_agent",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Rewrites and delivers the final response incorporating critique feedback.",
    instruction="""
    USER_QUERY: { PROMPT? }
    INITIAL_RESPONSE: { INITIAL_RESPONSE? }
    CRITIQUE_FEEDBACK: { CRITIQUE_FEEDBACK? }

    INSTRUCTIONS:
    - Rewrite and refine INITIAL_RESPONSE into a polished, complete final response incorporating CRITIQUE_FEEDBACK.
    - Directly present the final, fully detailed answer to the user in your response text.
    """,
    tools=[],
    before_model_callback=logging_before_callback,
)

# ============================================================================
# 3. Pipeline Assembly & Root Greeter Agent
# ============================================================================

answer_team = SequentialAgent(
    name="answer_team",
    description="Sequential pipeline: Search -> Critique -> Refine.",
    sub_agents=[
        search_agent,
        critique_agent,
        refine_agent,
    ],
)

greeter = Agent(
    name="greeter",
    model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
    description="Main entry agent routing user question to answer_team.",
    instruction="""
    INSTRUCTIONS:
    - Save the user's question into state field 'PROMPT' using `append_to_state`.
    - Transfer control to 'answer_team'.
    """,
    tools=[append_to_state],
    sub_agents=[answer_team],
    before_model_callback=logging_before_callback,
)



/tmp/ipykernel_99/439105134.py:131: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_team = SequentialAgent(


In [13]:
# ============================================================================
# 4. Execution and Runner Setup
# ============================================================================

APP_NAME = "verified_qa_app"
session_service = InMemorySessionService()

async def get_or_create_session(user_id: str, session_id: str):
    """Ensures a session exists before agent execution."""
    session = await session_service.get_session(
        app_name=APP_NAME, user_id=user_id, session_id=session_id
    )
    if not session:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=user_id, session_id=session_id
        )
    return session

def run_interactive_assistant(
    user_id: str = "qa_user",
    session_id: str = "session_001"
):
    """Runs an interactive QA session prompting the user for an open-ended question."""

    print("\n" + "="*60)
    print("🧠 VERIFIED QUESTION & ANSWER AGENT SYSTEM")
    print("="*60)

    user_query = input("👉 Enter your question: ").strip()

    if not user_query:
        user_query = "Find the next 3 golf events to occur in the next month"

    # Dynamically retrieve system time context
    current_time_context = datetime.now().strftime("%A, %B %d, %Y")

    # Inject runtime time context into prompt
    contextualized_prompt = (
        f"[SYSTEM CONTEXT: The current date is {current_time_context}. "
        f"Ensure all search queries and facts reflect events relative to this date.]\n\n"
        f"USER QUESTION: {user_query}"
    )

    print(f"\n🚀 Processing & Verifying Answer for: '{user_query}'...\n" + "="*60)

    # Initialize session
    loop = asyncio.get_event_loop()
    loop.run_until_complete(
        get_or_create_session(user_id=user_id, session_id=session_id)
    )

    runner = Runner(
        app_name=APP_NAME,
        agent=greeter,
        session_service=session_service,
    )

    formatted_message = Content(
        role="user",
        parts=[Part.from_text(text=contextualized_prompt)]
    )

    event_stream = runner.run(
        user_id=user_id,
        session_id=session_id,
        new_message=formatted_message,
    )

    for event in event_stream:
        if hasattr(event, "author") and event.author:
            print(f"\n🤖 [{event.author}]:")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "text") and part.text:
                    print(part.text)

    print("\n" + "="*60)
    print("✅ Answer Complete!")

# ============================================================================
# 5. Run Execution
# ============================================================================

run_interactive_assistant()


🧠 VERIFIED QUESTION & ANSWER AGENT SYSTEM
👉 Enter your question: Why is the sky blue?

🚀 Processing & Verifying Answer for: 'Why is the sky blue?'...

⚙️ [Callback] Invoking agent: greeter

🤖 [greeter]:

⚙️ [Callback] Invoking agent: greeter

🤖 [greeter]:

⚙️ [Callback] Invoking agent: search_agent
🤖 [greeter]:

🤖 [greeter]:


⚙️ [Callback] Invoking agent: search_sub_agent
🤖 [search_agent]:


⚙️ [Callback] Invoking agent: search_agent
🤖 [search_agent]:


🤖 [search_agent]:
⚙️ [Callback] Invoking agent: search_agent


🤖 [search_agent]:

⚙️ [Callback] Invoking agent: critique_agent
🤖 [search_agent]:
The sky is blue primarily due to a physical phenomenon called **Rayleigh scattering**, which describes how light interacts with tiny gas molecules in Earth's atmosphere, combined with the biology of human visual perception.

---

### 1. Sunlight is Made of Many Colors
Sunlight appears white to us, but it contains all the colors of the rainbow combined. Light travels in waves, and each color c

In [14]:
run_interactive_assistant()


🧠 VERIFIED QUESTION & ANSWER AGENT SYSTEM
👉 Enter your question: When do the Pittsburgh Steelers play this month?

🚀 Processing & Verifying Answer for: 'When do the Pittsburgh Steelers play this month?'...

⚙️ [Callback] Invoking agent: greeter

🤖 [greeter]:
⚙️ [Callback] Invoking agent: greeter


🤖 [greeter]:

🤖 [greeter]:
⚙️ [Callback] Invoking agent: search_agent


🤖 [greeter]:

⚙️ [Callback] Invoking agent: search_sub_agent
🤖 [search_agent]:


⚙️ [Callback] Invoking agent: search_agent
🤖 [search_agent]:


🤖 [search_agent]:
⚙️ [Callback] Invoking agent: search_sub_agent


⚙️ [Callback] Invoking agent: search_agent
🤖 [search_agent]:


⚙️ [Callback] Invoking agent: search_agent
🤖 [search_agent]:

🤖 [search_agent]:


⚙️ [Callback] Invoking agent: critique_agent
🤖 [search_agent]:
Here is the schedule for the **Pittsburgh Steelers** for **August 2026** (NFL Preseason):

---

### **Pittsburgh Steelers August 2026 Preseason Schedule**

1. **Preseason Week 1: vs. Green Bay Packers**
   * *